In [ ]:
# ============================================================
# CELL 1: IMPORT REQUIRED LIBRARIES
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# ============================================================
# CELL 2: DEVICE AND RANDOM SEED
# ============================================================

# Reproducibility:
# If we run the notebook several times, the random initialization
# will be more consistent when the same seed is used.
torch.manual_seed(42)

# Use GPU if available, otherwise CPU.
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

In [ ]:
# ============================================================
# CELL 3: LOAD THE MNIST DATASET
# ============================================================

# transforms.ToTensor():
#   Converts an MNIST image into a PyTorch tensor.
#
# Original pixel values:
#     0, 1, ..., 255
#
# After ToTensor():
#     values are scaled to [0, 1].
#
# We deliberately DO NOT normalize to negative values because
# we will use a Bernoulli-style reconstruction loss
# (Binary Cross Entropy).
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    transform=transform,
    download=True
)

print("Training examples:", len(train_dataset))
print("Test examples:", len(test_dataset))

In [ ]:
# ============================================================
# CELL 4: DATA LOADERS
# ============================================================

BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Number of training batches:", len(train_loader))
print("Number of test batches:", len(test_loader))

In [ ]:
# ============================================================
# CELL 5: VISUALIZE SOME MNIST IMAGES
# ============================================================

images, labels = next(iter(train_loader))

plt.figure(figsize=(10, 4))

for i in range(10):
    plt.subplot(2, 5, i + 1)

    # images[i] has shape:
    #     [1, 28, 28]
    #
    # squeeze removes the channel dimension.
    plt.imshow(images[i].squeeze(), cmap="gray")

    plt.title(f"Label: {labels[i].item()}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 7: DEFINE THE VARIATIONAL AUTOENCODER
# ============================================================

class VAE(nn.Module):

    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=20):
        super().__init__()

        # ----------------------------------------------------
        # ENCODER
        # ----------------------------------------------------
        #
        # Input:
        #     x -> 784 dimensional flattened MNIST image
        #
        # Hidden representation:
        #     784 -> 400
        self.fc1 = nn.Linear(input_dim, hidden_dim)

        # The encoder does NOT directly produce z.
        #
        # Instead, it produces parameters of q(z|x):
        #
        #     mu
        #     log(sigma^2)
        #
        # Therefore we need two separate output layers.

        self.fc_mu = nn.Linear(hidden_dim, latent_dim)

        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # ----------------------------------------------------
        # DECODER
        # ----------------------------------------------------

        # The decoder starts from latent z.
        self.fc2 = nn.Linear(latent_dim, hidden_dim)

        # Final reconstruction:
        #     hidden_dim -> 784
        self.fc3 = nn.Linear(hidden_dim, input_dim)


    def encode(self, x):
        """
        Encoder network.

        Given x, produce:
            mu
            logvar = log(sigma^2)

        These define:

            q(z|x) = N(mu, diag(sigma^2))
        """

        h = F.relu(self.fc1(x))

        mu = self.fc_mu(h)

        logvar = self.fc_logvar(h)

        return mu, logvar


    def reparameterize(self, mu, logvar):
        """
        REPARAMETERIZATION TRICK

        We want:

            z ~ N(mu, sigma^2)

        Direct sampling would make differentiation through
        the random operation difficult.

        Instead write:

            epsilon ~ N(0, I)

            z = mu + sigma * epsilon

        Since:

            logvar = log(sigma^2)

        we get:

            sigma^2 = exp(logvar)

        therefore:

            sigma = exp(0.5 * logvar)
        """

        std = torch.exp(0.5 * logvar)

        # Random noise epsilon ~ N(0, I)
        eps = torch.randn_like(std)

        # Reparameterized sample
        z = mu + std * eps

        return z


    def decode(self, z):
        """
        Decoder:

            z -> reconstructed x

        We use sigmoid at the output because MNIST pixels
        are in [0,1].
        """

        h = F.relu(self.fc2(z))

        x_recon = torch.sigmoid(self.fc3(h))

        return x_recon


    def forward(self, x):

        # Flatten:
        #
        # [batch, 1, 28, 28]
        #        ->
        # [batch, 784]
        x = x.view(x.size(0), -1)

        # Encoder
        mu, logvar = self.encode(x)

        # Sample z
        z = self.reparameterize(mu, logvar)

        # Decoder
        x_recon = self.decode(z)

        return x_recon, mu, logvar

In [ ]:
# ============================================================
# CELL 8: CREATE THE MODEL
# ============================================================

LATENT_DIM = 20

model = VAE(
    input_dim=784,
    hidden_dim=400,
    latent_dim=LATENT_DIM
).to(device)

print(model)

In [ ]:
# ============================================================
# CELL 11: VAE LOSS
# ============================================================

def vae_loss(recon_x, x, mu, logvar):

    # Flatten original x to match recon_x.
    x = x.view(x.size(0), -1)

    # --------------------------------------------------------
    # 1. RECONSTRUCTION LOSS
    # --------------------------------------------------------
    #
    # Decoder output lies in [0,1] because of sigmoid.
    #
    # For MNIST we can model each pixel approximately as
    # Bernoulli-distributed.
    #
    # BCE corresponds to negative log-likelihood under
    # this Bernoulli observation model.
    reconstruction_loss = F.binary_cross_entropy(
        recon_x,
        x,
        reduction="sum"
    )

    # --------------------------------------------------------
    # 2. KL DIVERGENCE
    # --------------------------------------------------------
    #
    # KL:
    #
    # q(z|x) = N(mu, diag(sigma^2))
    #
    # p(z) = N(0, I)
    #
    # Closed-form:
    #
    # 0.5 * sum(
    #     mu^2
    #     + sigma^2
    #     - log(sigma^2)
    #     - 1
    # )
    #
    # Since:
    #
    # logvar = log(sigma^2)
    #
    # sigma^2 = exp(logvar)

    kl_loss = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp()
    )

    # Total negative ELBO
    total_loss = reconstruction_loss + kl_loss

    return total_loss, reconstruction_loss, kl_loss

In [ ]:
# ============================================================
# CELL 13: OPTIMIZER
# ============================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [ ]:
# ============================================================
# CELL 14: TRAINING FUNCTION
# ============================================================

def train_one_epoch(model, loader, optimizer, device):

    model.train()

    total_loss = 0
    total_recon = 0
    total_kl = 0

    for images, _ in loader:

        images = images.to(device)

        # ----------------------------------------------------
        # Clear gradients from previous iteration
        # ----------------------------------------------------
        optimizer.zero_grad()

        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------
        recon_images, mu, logvar = model(images)

        # ----------------------------------------------------
        # Compute VAE loss
        # ----------------------------------------------------
        loss, recon_loss, kl_loss = vae_loss(
            recon_images,
            images,
            mu,
            logvar
        )

        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------
        loss.backward()

        # ----------------------------------------------------
        # Update parameters
        # ----------------------------------------------------
        optimizer.step()

        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()

    # Divide by number of examples rather than batches.
    #
    # Since loss was computed using reduction="sum",
    # this gives average loss per data example.
    N = len(loader.dataset)

    return (
        total_loss / N,
        total_recon / N,
        total_kl / N
    )

In [ ]:
# ============================================================
# CELL 15: TEST / VALIDATION FUNCTION
# ============================================================

def evaluate(model, loader, device):

    model.eval()

    total_loss = 0
    total_recon = 0
    total_kl = 0

    # No gradients are required during evaluation.
    with torch.no_grad():

        for images, _ in loader:

            images = images.to(device)

            recon_images, mu, logvar = model(images)

            loss, recon_loss, kl_loss = vae_loss(
                recon_images,
                images,
                mu,
                logvar
            )

            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl_loss.item()

    N = len(loader.dataset)

    return (
        total_loss / N,
        total_recon / N,
        total_kl / N
    )

In [ ]:
# ============================================================
# CELL 16: TRAIN THE VAE
# ============================================================

EPOCHS = 20

train_losses = []
test_losses = []

train_recon_losses = []
train_kl_losses = []

for epoch in range(1, EPOCHS + 1):

    train_loss, train_recon, train_kl = train_one_epoch(
        model,
        train_loader,
        optimizer,
        device
    )

    test_loss, test_recon, test_kl = evaluate(
        model,
        test_loader,
        device
    )

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    train_recon_losses.append(train_recon)
    train_kl_losses.append(train_kl)

    print(
        f"Epoch [{epoch:02d}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Recon: {train_recon:.4f} | "
        f"KL: {train_kl:.4f} | "
        f"Test Loss: {test_loss:.4f}"
    )

In [ ]:
# ============================================================
# CELL 17: TOTAL LOSS CURVE
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    range(1, EPOCHS + 1),
    train_losses,
    label="Train"
)

plt.plot(
    range(1, EPOCHS + 1),
    test_losses,
    label="Test"
)

plt.xlabel("Epoch")
plt.ylabel("Negative ELBO")
plt.title("VAE Training Curve")
plt.legend()
plt.show()

In [ ]:
# ============================================================
# CELL 18: RECONSTRUCTION AND KL LOSS
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    range(1, EPOCHS + 1),
    train_recon_losses,
    label="Reconstruction Loss"
)

plt.plot(
    range(1, EPOCHS + 1),
    train_kl_losses,
    label="KL Divergence"
)

plt.xlabel("Epoch")
plt.ylabel("Loss per example")
plt.title("Reconstruction Loss vs KL Loss")
plt.legend()

plt.show()

In [ ]:
# ============================================================
# CELL 19: ORIGINAL VS RECONSTRUCTED MNIST
# ============================================================

model.eval()

images, labels = next(iter(test_loader))

images = images[:10].to(device)

with torch.no_grad():
    reconstructed, mu, logvar = model(images)

# Convert outputs back to image shape
reconstructed = reconstructed.view(-1, 1, 28, 28)

images = images.cpu()
reconstructed = reconstructed.cpu()

plt.figure(figsize=(15, 4))

for i in range(10):

    # Original images
    plt.subplot(2, 10, i + 1)

    plt.imshow(
        images[i].squeeze(),
        cmap="gray"
    )

    plt.axis("off")

    if i == 0:
        plt.ylabel("Original")


    # Reconstructed images
    plt.subplot(2, 10, 10 + i + 1)

    plt.imshow(
        reconstructed[i].squeeze(),
        cmap="gray"
    )

    plt.axis("off")

    if i == 0:
        plt.ylabel("Recon")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 20: GENERATE NEW MNIST DIGITS
# ============================================================

model.eval()

NUM_SAMPLES = 20

# Sample directly from the prior:
#
# z ~ N(0, I)
z = torch.randn(
    NUM_SAMPLES,
    LATENT_DIM,
    device=device
)

with torch.no_grad():

    generated = model.decode(z)

generated = generated.view(
    -1,
    1,
    28,
    28
).cpu()

plt.figure(figsize=(12, 5))

for i in range(NUM_SAMPLES):

    plt.subplot(4, 5, i + 1)

    plt.imshow(
        generated[i].squeeze(),
        cmap="gray"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 21: INSPECT THE LATENT DISTRIBUTION
# ============================================================

model.eval()

images, labels = next(iter(test_loader))

images = images[:5].to(device)

with torch.no_grad():

    flat_images = images.view(images.size(0), -1)

    mu, logvar = model.encode(flat_images)

    std = torch.exp(0.5 * logvar)

    z = model.reparameterize(mu, logvar)

print("Shape of mu:")
print(mu.shape)

print("\nFirst example mu:")
print(mu[0])

print("\nFirst example standard deviation:")
print(std[0])

print("\nSampled latent vector z:")
print(z[0])

In [ ]:
# ============================================================
# CELL 23: REPARAMETERIZATION NUMERICAL EXAMPLE
# ============================================================

mu_demo = 2.0
sigma_demo = 0.5

N = 100000

epsilon = torch.randn(N)

z_samples = mu_demo + sigma_demo * epsilon

print(
    "Empirical mean:",
    z_samples.mean().item()
)

print(
    "Empirical standard deviation:",
    z_samples.std().item()
)

In [ ]:
# ============================================================
# CELL 24: HISTOGRAM OF REPARAMETERIZED SAMPLES
# ============================================================

plt.figure(figsize=(8, 5))

plt.hist(
    z_samples.numpy(),
    bins=70,
    density=True
)

plt.xlabel("z")
plt.ylabel("Density")
plt.title("Samples from N(2, 0.5²) using Reparameterization")

plt.show()

In [ ]:
# ============================================================
# CELL 26: TRAIN A 2-D LATENT VAE
# ============================================================

LATENT_DIM_2D = 2

model_2d = VAE(
    input_dim=784,
    hidden_dim=400,
    latent_dim=LATENT_DIM_2D
).to(device)

optimizer_2d = optim.Adam(
    model_2d.parameters(),
    lr=1e-3
)

EPOCHS_2D = 15

for epoch in range(1, EPOCHS_2D + 1):

    train_loss, recon_loss, kl_loss = train_one_epoch(
        model_2d,
        train_loader,
        optimizer_2d,
        device
    )

    print(
        f"Epoch {epoch:02d} | "
        f"Loss: {train_loss:.3f} | "
        f"Recon: {recon_loss:.3f} | "
        f"KL: {kl_loss:.3f}"
    )

In [ ]:
# ============================================================
# CELL 27: VISUALIZE 2-D LATENT REPRESENTATIONS
# ============================================================

model_2d.eval()

all_mu = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        images_flat = images.view(
            images.size(0),
            -1
        )

        mu, logvar = model_2d.encode(images_flat)

        # For visualization, use the mean mu rather than
        # randomly sampled z to avoid sampling noise.
        all_mu.append(mu.cpu())

        all_labels.append(labels)

all_mu = torch.cat(all_mu).numpy()
all_labels = torch.cat(all_labels).numpy()

plt.figure(figsize=(10, 8))

scatter = plt.scatter(
    all_mu[:, 0],
    all_mu[:, 1],
    c=all_labels,
    s=5
)

plt.colorbar(scatter)

plt.xlabel("Latent dimension z1")
plt.ylabel("Latent dimension z2")

plt.title("MNIST in the Learned 2-D VAE Latent Space")

plt.show()

In [ ]:
# ============================================================
# CELL 28: GENERATE A LATENT SPACE MANIFOLD
# ============================================================

model_2d.eval()

GRID_SIZE = 20

# Sample points ranging approximately from -3 to 3.
#
# Most of the probability mass of N(0,1) lies inside
# approximately [-3, 3].
grid_x = np.linspace(-3, 3, GRID_SIZE)

grid_y = np.linspace(-3, 3, GRID_SIZE)

canvas = np.zeros(
    (28 * GRID_SIZE, 28 * GRID_SIZE)
)

with torch.no_grad():

    for i, y in enumerate(grid_y):

        for j, x in enumerate(grid_x):

            z = torch.tensor(
                [[x, y]],
                dtype=torch.float32,
                device=device
            )

            generated = model_2d.decode(z)

            digit = generated.view(
                28,
                28
            ).cpu().numpy()

            canvas[
                i * 28:(i + 1) * 28,
                j * 28:(j + 1) * 28
            ] = digit

plt.figure(figsize=(12, 12))

plt.imshow(
    canvas,
    cmap="gray"
)

plt.xlabel("Latent z1")
plt.ylabel("Latent z2")
plt.title("MNIST VAE Latent Manifold")

plt.show()

In [ ]:
# ============================================================
# CELL 30: LATENT INTERPOLATION BETWEEN TWO MNIST IMAGES
# ============================================================

#

model.eval()

images, labels = next(iter(test_loader))

# Choose two images
image_A = images[0:1].to(device)
image_B = images[1:2].to(device)

label_A = labels[0].item()
label_B = labels[1].item()

with torch.no_grad():

    # Flatten
    flat_A = image_A.view(1, -1)
    flat_B = image_B.view(1, -1)

    # Encode
    mu_A, _ = model.encode(flat_A)
    mu_B, _ = model.encode(flat_B)

    # Number of interpolation steps
    NUM_STEPS = 12

    interpolated_images = []

    for alpha in torch.linspace(
        0,
        1,
        NUM_STEPS,
        device=device
    ):

        # Linear interpolation
        z = (
            (1 - alpha) * mu_A
            +
            alpha * mu_B
        )

        generated = model.decode(z)

        generated = generated.view(
            1,
            28,
            28
        )

        interpolated_images.append(
            generated.cpu()
        )


plt.figure(figsize=(18, 3))

for i, img in enumerate(interpolated_images):

    plt.subplot(
        1,
        NUM_STEPS,
        i + 1
    )

    plt.imshow(
        img.squeeze(),
        cmap="gray"
    )

    plt.axis("off")

plt.suptitle(
    f"Interpolation: Digit {label_A} → Digit {label_B}"
)

plt.show()

In [ ]:
# ============================================================
# CELL 31: DETERMINISTIC RECONSTRUCTION USING MU
# ============================================================

model.eval()

images, labels = next(iter(test_loader))

images = images[:10].to(device)

with torch.no_grad():

    flat = images.view(images.size(0), -1)

    mu, logvar = model.encode(flat)

    # Instead of sampling z,
    # directly use the posterior mean.
    z = mu

    reconstructed = model.decode(z)

reconstructed = reconstructed.view(
    -1,
    1,
    28,
    28
).cpu()

images = images.cpu()

plt.figure(figsize=(15, 4))

for i in range(10):

    plt.subplot(2, 10, i + 1)

    plt.imshow(
        images[i].squeeze(),
        cmap="gray"
    )

    plt.axis("off")

    plt.subplot(2, 10, i + 11)

    plt.imshow(
        reconstructed[i].squeeze(),
        cmap="gray"
    )

    plt.axis("off")

plt.show()

In [ ]:
# ============================================================
# CELL 32: PRIOR VS POSTERIOR SAMPLING
# ============================================================

model.eval()

images, labels = next(iter(test_loader))
images = images[:10].to(device)

with torch.no_grad():

    # --------------------------------------------------------
    # POSTERIOR SAMPLES
    #
    # z ~ q(z|x)
    # --------------------------------------------------------

    flat = images.view(images.size(0), -1)

    mu, logvar = model.encode(flat)

    z_posterior = model.reparameterize(
        mu,
        logvar
    )

    posterior_generated = model.decode(
        z_posterior
    )


    # --------------------------------------------------------
    # PRIOR SAMPLES
    #
    # z ~ N(0,I)
    # --------------------------------------------------------

    z_prior = torch.randn(
        10,
        LATENT_DIM,
        device=device
    )

    prior_generated = model.decode(
        z_prior
    )

In [ ]:
# ============================================================
# CELL 33: DISPLAY PRIOR AND POSTERIOR GENERATED IMAGES
# ============================================================

# posterior -> parameter from the encoder
# Prior - > sample from normal 0, I

posterior_generated = posterior_generated.view(
    -1,
    1,
    28,
    28
).cpu()

prior_generated = prior_generated.view(
    -1,
    1,
    28,
    28
).cpu()

plt.figure(figsize=(15, 4))

for i in range(10):

    plt.subplot(2, 10, i + 1)

    plt.imshow(
        posterior_generated[i].squeeze(),
        cmap="gray"
    )

    plt.axis("off")

    if i == 0:
        plt.ylabel("Posterior")


    plt.subplot(2, 10, 10 + i + 1)

    plt.imshow(
        prior_generated[i].squeeze(),
        cmap="gray"
    )

    plt.axis("off")

    if i == 0:
        plt.ylabel("Prior")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 34: LATENT STATISTICS
# ============================================================

# Aggregate mean = 0, std = 1

model.eval()

all_mu = []
all_logvar = []

with torch.no_grad():

    for images, _ in test_loader:

        images = images.to(device)

        flat = images.view(
            images.size(0),
            -1
        )

        mu, logvar = model.encode(flat)

        all_mu.append(mu.cpu())
        all_logvar.append(logvar.cpu())

all_mu = torch.cat(all_mu)
all_logvar = torch.cat(all_logvar)

all_var = torch.exp(all_logvar)

print(
    "Average posterior mean:",
    all_mu.mean().item()
)

print(
    "Average posterior variance:",
    all_var.mean().item()
)

In [ ]:
# VQ-VAE

# ============================================================
# CELL 1: IMPORT REQUIRED LIBRARIES
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# ============================================================
# CELL 2: DEVICE AND RANDOM SEED
# ============================================================

# Fixing the random seed helps make experiments more reproducible.
torch.manual_seed(42)

# Use GPU when available.
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

In [ ]:
# ============================================================
# CELL 3: LOAD MNIST DATASET
# ============================================================

# transforms.ToTensor() converts images from:
#
#     [0, 255]
#
# to:
#
#     [0, 1]

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    transform=transform,
    download=True
)

print("Training samples:", len(train_dataset))
print("Testing samples :", len(test_dataset))

In [ ]:
# ============================================================
# CELL 4: CREATE DATA LOADERS
# ============================================================

BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Testing batches :", len(test_loader))

In [ ]:
# ============================================================
# CELL 5: VISUALIZE SOME MNIST IMAGES
# ============================================================

images, labels = next(iter(train_loader))

plt.figure(figsize=(10, 4))

for i in range(10):

    plt.subplot(2, 5, i + 1)

    plt.imshow(
        images[i].squeeze(),
        cmap="gray"
    )

    plt.title(
        f"Label: {labels[i].item()}"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 6: VQ-VAE HYPERPARAMETERS
# ============================================================

# Number of channels produced by the encoder before quantization.
#
# Each latent vector will have this many dimensions.
EMBEDDING_DIM = 64

# Number of vectors in the codebook.
#
# Think of this as vocabulary size.
NUM_EMBEDDINGS = 128

# Commitment loss coefficient beta.
BETA = 0.25

# Training settings
LEARNING_RATE = 1e-3

EPOCHS = 20

In [ ]:
# ============================================================
# CELL 7: DEFINE THE ENCODER
# ============================================================

class Encoder(nn.Module):

    def __init__(self, embedding_dim=64):

        super().__init__()

        self.encoder = nn.Sequential(

            # ------------------------------------------------
            # INPUT
            #
            # [B, 1, 28, 28]
            # ------------------------------------------------

            nn.Conv2d(
                in_channels=1,
                out_channels=32,
                kernel_size=4,
                stride=2,
                padding=1
            ),

            nn.ReLU(),

            # Shape:
            #
            # [B, 32, 14, 14]


            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=4,
                stride=2,
                padding=1
            ),

            nn.ReLU(),

            # Shape:
            #
            # [B, 64, 7, 7]


            nn.Conv2d(
                in_channels=64,
                out_channels=embedding_dim,
                kernel_size=3,
                stride=1,
                padding=1
            )

            # Final shape:
            #
            # [B, embedding_dim, 7, 7]
        )


    def forward(self, x):

        return self.encoder(x)

In [ ]:
# ============================================================
# CELL 8: CHECK ENCODER OUTPUT
# ============================================================

encoder_test = Encoder(
    embedding_dim=EMBEDDING_DIM
).to(device)

images, _ = next(iter(train_loader))

images = images.to(device)

with torch.no_grad():

    z_e = encoder_test(images)

print("Input shape :", images.shape)
print("Encoder shape:", z_e.shape)

In [ ]:
# ============================================================
# CELL 9: VECTOR QUANTIZATION MODULE
# ============================================================

class VectorQuantizer(nn.Module):

    def __init__(
        self,
        num_embeddings,
        embedding_dim,
        beta=0.25
    ):

        super().__init__()

        self.num_embeddings = num_embeddings

        self.embedding_dim = embedding_dim

        self.beta = beta


        # ----------------------------------------------------
        # CODEBOOK
        # ----------------------------------------------------
        #
        # This is essentially a learnable lookup table.
        #
        # Shape:
        #
        #     [K, D]
        #
        # where:
        #
        #     K = number of codebook vectors
        #
        #     D = dimension of each vector

        self.embedding = nn.Embedding(
            num_embeddings,
            embedding_dim
        )


        # Initialize embeddings with small random values.
        #
        # A common choice is uniform initialization.

        self.embedding.weight.data.uniform_(
            -1.0 / num_embeddings,
            1.0 / num_embeddings
        )


    def forward(self, z_e):

        # ----------------------------------------------------
        # z_e shape:
        #
        # [B, D, H, W]
        #
        # We need vectors of dimension D to appear in the
        # last dimension.
        #
        # Convert:
        #
        # [B, D, H, W]
        #
        # ->
        #
        # [B, H, W, D]
        # ----------------------------------------------------

        z_e_permuted = z_e.permute(
            0,
            2,
            3,
            1
        ).contiguous()


        # ----------------------------------------------------
        # Flatten all spatial latent vectors.
        #
        # [B, H, W, D]
        #
        # ->
        #
        # [B*H*W, D]
        #
        # Each row is now one encoder latent vector.
        # ----------------------------------------------------

        flat_z_e = z_e_permuted.view(
            -1,
            self.embedding_dim
        )


        # ----------------------------------------------------
        # COMPUTE DISTANCES TO EVERY CODEBOOK VECTOR
        # ----------------------------------------------------
        #
        # We want:
        #
        # ||z - e||^2
        #
        # for every encoder vector z
        # and every embedding e.
        #
        # Expand:
        #
        # ||z - e||^2
        #
        # =
        #
        # ||z||^2
        #
        # +
        #
        # ||e||^2
        #
        # -
        #
        # 2 z^T e
        #
        #
        # This avoids explicitly constructing huge
        # difference tensors.
        # ----------------------------------------------------

        distances = (

            torch.sum(
                flat_z_e ** 2,
                dim=1,
                keepdim=True
            )

            +

            torch.sum(
                self.embedding.weight ** 2,
                dim=1
            )

            -

            2
            *
            torch.matmul(
                flat_z_e,
                self.embedding.weight.t()
            )
        )


        # distances shape:
        #
        # [B*H*W, K]


        # ----------------------------------------------------
        # FIND CLOSEST CODEBOOK VECTOR
        # ----------------------------------------------------
        #
        # For each encoder vector:
        #
        # k* = argmin_k ||z_e - e_k||^2
        # ----------------------------------------------------

        encoding_indices = torch.argmin(
            distances,
            dim=1
        )


        # Shape:
        #
        # [B*H*W]


        # ----------------------------------------------------
        # LOOK UP THE SELECTED CODEBOOK VECTORS
        # ----------------------------------------------------

        z_q = self.embedding(
            encoding_indices
        )


        # Shape:
        #
        # [B*H*W, D]


        # ----------------------------------------------------
        # RESTORE SPATIAL SHAPE
        # ----------------------------------------------------

        z_q = z_q.view(
            z_e_permuted.shape
        )

        # Shape:
        #
        # [B, H, W, D]


        # ----------------------------------------------------
        # VQ LOSS
        # ----------------------------------------------------
        #
        # VQ-VAE uses TWO losses:
        #
        # 1. Codebook loss
        #
        # 2. Commitment loss
        # ----------------------------------------------------


        # ----------------------------------------------------
        # CODEBOOK LOSS
        #
        # ||sg[z_e] - z_q||^2
        #
        # sg = stop-gradient.
        #
        # We detach z_e so gradients update ONLY the codebook.
        # ----------------------------------------------------

        codebook_loss = F.mse_loss(
            z_q,
            z_e_permuted.detach()
        )


        # ----------------------------------------------------
        # COMMITMENT LOSS
        #
        # beta * ||z_e - sg[z_q]||^2
        #
        # Here z_q is detached.
        #
        # Therefore this term updates the ENCODER,
        # encouraging its outputs to stay close to
        # selected codebook vectors.
        # ----------------------------------------------------

        commitment_loss = self.beta * F.mse_loss(
            z_e_permuted,
            z_q.detach()
        )


        # Total vector quantization loss
        vq_loss = (
            codebook_loss
            +
            commitment_loss
        )


        # ----------------------------------------------------
        # STRAIGHT-THROUGH ESTIMATOR
        # ----------------------------------------------------
        #
        # Quantization uses argmin.
        #
        # argmin is not differentiable.
        #
        # We want the decoder to receive z_q,
        # but during backpropagation we want the gradient
        # to flow approximately as if quantization were
        # the identity operation.
        #
        # Use:
        #
        # z_q_st =
        #
        # z_e
        #
        # +
        #
        # stop_gradient(z_q - z_e)
        #
        #
        # Forward:
        #
        #     z_q_st = z_q
        #
        # Backward:
        #
        #     dz_q_st/dz_e = 1
        # ----------------------------------------------------

        z_q_st = (
            z_e_permuted
            +
            (
                z_q
                -
                z_e_permuted
            ).detach()
        )


        # ----------------------------------------------------
        # Convert back:
        #
        # [B, H, W, D]
        #
        # ->
        #
        # [B, D, H, W]
        # ----------------------------------------------------

        z_q_st = z_q_st.permute(
            0,
            3,
            1,
            2
        ).contiguous()


        return (
            z_q_st,
            vq_loss,
            encoding_indices,
            codebook_loss,
            commitment_loss
        )

In [ ]:
# ============================================================
# CELL 10: DEFINE THE DECODER
# ============================================================

class Decoder(nn.Module):

    def __init__(self, embedding_dim=64):

        super().__init__()

        self.decoder = nn.Sequential(

            # ------------------------------------------------
            # Input:
            #
            # [B, D, 7, 7]
            # ------------------------------------------------

            nn.ConvTranspose2d(
                in_channels=embedding_dim,
                out_channels=64,
                kernel_size=3,
                stride=1,
                padding=1
            ),

            nn.ReLU(),

            # Shape:
            #
            # [B, 64, 7, 7]


            nn.ConvTranspose2d(
                in_channels=64,
                out_channels=32,
                kernel_size=4,
                stride=2,
                padding=1
            ),

            nn.ReLU(),

            # Shape:
            #
            # [B, 32, 14, 14]


            nn.ConvTranspose2d(
                in_channels=32,
                out_channels=1,
                kernel_size=4,
                stride=2,
                padding=1
            ),

            # Shape:
            #
            # [B, 1, 28, 28]


            # Output pixels in [0,1]
            nn.Sigmoid()
        )


    def forward(self, z):

        return self.decoder(z)

In [ ]:
# ============================================================
# CELL 11: COMPLETE VQ-VAE
# ============================================================

class VQVAE(nn.Module):

    def __init__(
        self,
        num_embeddings=128,
        embedding_dim=64,
        beta=0.25
    ):

        super().__init__()


        # Encoder:
        #
        # x -> z_e
        self.encoder = Encoder(
            embedding_dim=embedding_dim
        )


        # Vector quantizer:
        #
        # z_e -> z_q
        self.quantizer = VectorQuantizer(
            num_embeddings=num_embeddings,
            embedding_dim=embedding_dim,
            beta=beta
        )


        # Decoder:
        #
        # z_q -> reconstructed image
        self.decoder = Decoder(
            embedding_dim=embedding_dim
        )


    def forward(self, x):

        # ----------------------------------------------------
        # STEP 1: ENCODE
        # ----------------------------------------------------

        z_e = self.encoder(x)


        # ----------------------------------------------------
        # STEP 2: QUANTIZE
        # ----------------------------------------------------

        (
            z_q,
            vq_loss,
            encoding_indices,
            codebook_loss,
            commitment_loss
        ) = self.quantizer(z_e)


        # ----------------------------------------------------
        # STEP 3: DECODE
        # ----------------------------------------------------

        reconstruction = self.decoder(
            z_q
        )


        return (
            reconstruction,
            vq_loss,
            encoding_indices,
            codebook_loss,
            commitment_loss,
            z_e,
            z_q
        )

In [ ]:
# ============================================================
# CELL 12: CREATE VQ-VAE MODEL
# ============================================================

model = VQVAE(
    num_embeddings=NUM_EMBEDDINGS,
    embedding_dim=EMBEDDING_DIM,
    beta=BETA
).to(device)

print(model)

In [ ]:
# ============================================================
# CELL 13: VQ-VAE LOSS
# ============================================================

def vqvae_loss(
    reconstruction,
    original,
    vq_loss
);

 # --------------------------------------------------------
    # RECONSTRUCTION LOSS
    # --------------------------------------------------------
    #
    # Measures how well the decoder reconstructs x.
    #
    # For MNIST, MSE is sufficient for demonstration.

    reconstruction_loss = F.mse_loss(
        reconstruction,
        original
    )


    # --------------------------------------------------------
    # TOTAL LOSS
    # --------------------------------------------------------

    total_loss = (
        reconstruction_loss
        +
        vq_loss
    )


    return (
        total_loss,
        reconstruction_loss
    )




In [ ]:
# ============================================================
# CELL 14: OPTIMIZER
# ============================================================

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

In [ ]:
# ============================================================
# CELL 15: ONE TRAINING STEP
# ============================================================

images, _ = next(iter(train_loader))

images = images.to(device)

# Clear gradients
optimizer.zero_grad()

# Forward pass
(
    reconstruction,
    vq_loss,
    encoding_indices,
    codebook_loss,
    commitment_loss,
    z_e,
    z_q
) = model(images)

# Calculate total loss
loss, reconstruction_loss = vqvae_loss(
    reconstruction,
    images,
    vq_loss
)

# Backpropagation
loss.backward()

# Parameter update
optimizer.step()

print(
    "Total loss:",
    loss.item()
)

print(
    "Reconstruction loss:",
    reconstruction_loss.item()
)

print(
    "Codebook loss:",
    codebook_loss.item()
)

print(
    "Commitment loss:",
    commitment_loss.item()
)

In [ ]:
# ============================================================
# CELL 16: TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    device
):

    model.train()


    total_loss = 0.0

    total_reconstruction = 0.0

    total_codebook = 0.0

    total_commitment = 0.0


    for images, _ in loader:

        images = images.to(device)


        # ----------------------------------------------------
        # CLEAR OLD GRADIENTS
        # ----------------------------------------------------

        optimizer.zero_grad()


        # ----------------------------------------------------
        # FORWARD PASS
        # ----------------------------------------------------

        (
            reconstruction,
            vq_loss,
            encoding_indices,
            codebook_loss,
            commitment_loss,
            z_e,
            z_q
        ) = model(images)


        # ----------------------------------------------------
        # LOSS
        # ----------------------------------------------------

        loss, reconstruction_loss = vqvae_loss(
            reconstruction,
            images,
            vq_loss
        )


        # ----------------------------------------------------
        # BACKPROPAGATION
        # ----------------------------------------------------

        loss.backward()


        # ----------------------------------------------------
        # UPDATE PARAMETERS
        # ----------------------------------------------------

        optimizer.step()


        # ----------------------------------------------------
        # STORE LOSSES
        # ----------------------------------------------------

        total_loss += loss.item()

        total_reconstruction += (
            reconstruction_loss.item()
        )

        total_codebook += (
            codebook_loss.item()
        )

        total_commitment += (
            commitment_loss.item()
        )


    number_batches = len(loader)


    return (

        total_loss / number_batches,

        total_reconstruction / number_batches,

        total_codebook / number_batches,

        total_commitment / number_batches
    )

In [ ]:
# ============================================================
# CELL 17: EVALUATION FUNCTION
# ============================================================

def evaluate(
    model,
    loader,
    device
):

    model.eval()


    total_loss = 0.0

    total_reconstruction = 0.0

    total_codebook = 0.0

    total_commitment = 0.0


    with torch.no_grad():

        for images, _ in loader:

            images = images.to(device)


            (
                reconstruction,
                vq_loss,
                encoding_indices,
                codebook_loss,
                commitment_loss,
                z_e,
                z_q
            ) = model(images)


            loss, reconstruction_loss = vqvae_loss(
                reconstruction,
                images,
                vq_loss
            )


            total_loss += loss.item()

            total_reconstruction += (
                reconstruction_loss.item()
            )

            total_codebook += (
                codebook_loss.item()
            )

            total_commitment += (
                commitment_loss.item()
            )


    number_batches = len(loader)


    return (

        total_loss / number_batches,

        total_reconstruction / number_batches,

        total_codebook / number_batches,

        total_commitment / number_batches
    )

In [ ]:
# ============================================================
# CELL 18: TRAIN VQ-VAE
# ============================================================

train_total_losses = []

train_recon_losses = []

train_codebook_losses = []

train_commitment_losses = []

test_total_losses = []


for epoch in range(
    1,
    EPOCHS + 1
):


    (
        train_loss,
        train_reconstruction,
        train_codebook,
        train_commitment
    ) = train_one_epoch(

        model,
        train_loader,
        optimizer,
        device
    )


    (
        test_loss,
        test_reconstruction,
        test_codebook,
        test_commitment
    ) = evaluate(

        model,
        test_loader,
        device
    )


    train_total_losses.append(
        train_loss
    )

    train_recon_losses.append(
        train_reconstruction
    )

    train_codebook_losses.append(
        train_codebook
    )

    train_commitment_losses.append(
        train_commitment
    )

    test_total_losses.append(
        test_loss
    )


    print(
        f"Epoch [{epoch:02d}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.5f} | "
        f"Recon: {train_reconstruction:.5f} | "
        f"Codebook: {train_codebook:.5f} | "
        f"Commitment: {train_commitment:.5f} | "
        f"Test Loss: {test_loss:.5f}"
    )

In [ ]:
# ============================================================
# CELL 19: TOTAL TRAINING LOSS
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    range(1, EPOCHS + 1),
    train_total_losses,
    label="Train Loss"
)

plt.plot(
    range(1, EPOCHS + 1),
    test_total_losses,
    label="Test Loss"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title(
    "VQ-VAE Total Loss"
)

plt.legend()

plt.show()

In [ ]:
# ============================================================
# CELL 20: LOSS COMPONENTS
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    range(1, EPOCHS + 1),
    train_recon_losses,
    label="Reconstruction"
)

plt.plot(
    range(1, EPOCHS + 1),
    train_codebook_losses,
    label="Codebook"
)

plt.plot(
    range(1, EPOCHS + 1),
    train_commitment_losses,
    label="Commitment"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title(
    "VQ-VAE Loss Components"
)

plt.legend()

plt.show()

In [ ]:
# ============================================================
# CELL 21: ORIGINAL VS RECONSTRUCTION
# ============================================================

model.eval()

images, labels = next(iter(test_loader))

images = images[:10].to(device)


with torch.no_grad():

    (
        reconstruction,
        vq_loss,
        encoding_indices,
        codebook_loss,
        commitment_loss,
        z_e,
        z_q
    ) = model(images)


images = images.cpu()

reconstruction = reconstruction.cpu()


plt.figure(figsize=(15, 4))


for i in range(10):

    # --------------------------------------------------------
    # Original
    # --------------------------------------------------------

    plt.subplot(
        2,
        10,
        i + 1
    )

    plt.imshow(
        images[i].squeeze(),
        cmap="gray"
    )

    plt.axis("off")

    if i == 0:
        plt.ylabel("Original")


    # --------------------------------------------------------
    # Reconstruction
    # --------------------------------------------------------

    plt.subplot(
        2,
        10,
        i + 11
    )

    plt.imshow(
        reconstruction[i].squeeze(),
        cmap="gray"
    )

    plt.axis("off")

    if i == 0:
        plt.ylabel("VQ-VAE")


plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# CELL 22: DISPLAY DISCRETE LATENT CODES
# ============================================================

model.eval()

images, labels = next(iter(test_loader))

image = images[0:1].to(device)


with torch.no_grad():

    (
        reconstruction,
        vq_loss,
        encoding_indices,
        codebook_loss,
        commitment_loss,
        z_e,
        z_q
    ) = model(image)


# Encoder spatial size is 7 x 7.
#
# encoding_indices currently has shape:
#
# [B * H * W]
#
# For B = 1:
#
# [49]

indices_grid = encoding_indices.view(
    7,
    7
).cpu()


print(
    "Digit label:",
    labels[0].item()
)

print(
    "\nDiscrete latent code indices:"
)

print(
    indices_grid
)

In [ ]:
# ============================================================
# CELL 23: VISUALIZE THE 7x7 DISCRETE LATENT MAP
# ============================================================

plt.figure(figsize=(5, 5))

plt.imshow(
    indices_grid,
    interpolation="nearest"
)

plt.colorbar(
    label="Codebook Index"
)

plt.title(
    "Discrete VQ-VAE Latent Representation"
)

plt.xlabel("Latent width")

plt.ylabel("Latent height")

plt.show()

In [ ]:
# ============================================================
# CELL 24: INSPECT LEARNED CODEBOOK
# ============================================================

codebook = (
    model.quantizer.embedding.weight
    .detach()
    .cpu()
)

print(
    "Codebook shape:",
    codebook.shape
)

print(
    "\nFirst codebook vector:"
)

print(
    codebook[0]
)

In [ ]:
# ============================================================
# CELL 25: MEASURE CODEBOOK USAGE
# ============================================================

model.eval()

all_indices = []


with torch.no_grad():

    for images, _ in test_loader:

        images = images.to(device)

        (
            reconstruction,
            vq_loss,
            encoding_indices,
            codebook_loss,
            commitment_loss,
            z_e,
            z_q
        ) = model(images)

        all_indices.append(
            encoding_indices.cpu()
        )


all_indices = torch.cat(
    all_indices
)


unique_indices = torch.unique(
    all_indices
)


print(
    "Total codebook size:",
    NUM_EMBEDDINGS
)

print(
    "Number of used embeddings:",
    len(unique_indices)
)

print(
    "Percentage used:",
    100
    *
    len(unique_indices)
    /
    NUM_EMBEDDINGS
)

In [ ]:
# ============================================================
# CELL 26: CODEBOOK USAGE HISTOGRAM
# ============================================================

counts = torch.bincount(
    all_indices,
    minlength=NUM_EMBEDDINGS
)


plt.figure(figsize=(12, 5))

plt.bar(
    np.arange(NUM_EMBEDDINGS),
    counts.numpy()
)

plt.xlabel(
    "Codebook Index"
)

plt.ylabel(
    "Number of Assignments"
)

plt.title(
    "VQ-VAE Codebook Usage"
)

plt.show()

In [ ]:
# ============================================================
# CELL 27: CODEBOOK PERPLEXITY
# ============================================================

counts = torch.bincount(
    all_indices,
    minlength=NUM_EMBEDDINGS
).float()


probabilities = counts / counts.sum()


# Remove zero-probability entries to avoid:
#
# log(0)

nonzero_probabilities = probabilities[
    probabilities > 0
]


entropy = -torch.sum(

    nonzero_probabilities
    *
    torch.log(
        nonzero_probabilities
    )
)


perplexity = torch.exp(
    entropy
)


print(
    "Codebook perplexity:",
    perplexity.item()
)

print(
    "Maximum possible perplexity:",
    NUM_EMBEDDINGS
)

In [ ]:
# ============================================================
# CELL 28: DECODE DIRECTLY FROM CODEBOOK INDICES
# ============================================================

model.eval()

images, labels = next(iter(test_loader))

image = images[0:1].to(device)


with torch.no_grad():

    # --------------------------------------------------------
    # Encode
    # --------------------------------------------------------

    z_e = model.encoder(
        image
    )


    # --------------------------------------------------------
    # Quantize
    # --------------------------------------------------------

    (
        z_q,
        vq_loss,
        encoding_indices,
        codebook_loss,
        commitment_loss
    ) = model.quantizer(
        z_e
    )


    # --------------------------------------------------------
    # Recover embeddings from integer indices
    # --------------------------------------------------------

    quantized_vectors = (
        model.quantizer.embedding(
            encoding_indices
        )
    )


    # Shape:
    #
    # [49, 64]
    #
    # Convert:
    #
    # [49,64]
    #
    # ->
    #
    # [1,7,7,64]

    quantized_vectors = quantized_vectors.view(
        1,
        7,
        7,
        EMBEDDING_DIM
    )


    # Convert:
    #
    # [B,H,W,D]
    #
    # ->
    #
    # [B,D,H,W]

    quantized_vectors = quantized_vectors.permute(
        0,
        3,
        1,
        2
    )


    # Decode
    reconstruction_from_indices = (
        model.decoder(
            quantized_vectors
        )
    )

In [ ]:
# ============================================================
# CELL 29: SHOW RECONSTRUCTION FROM DISCRETE CODES
# ============================================================

plt.figure(figsize=(8, 4))


plt.subplot(1, 2, 1)

plt.imshow(
    image[0].cpu().squeeze(),
    cmap="gray"
)

plt.title(
    "Original"
)

plt.axis("off")


plt.subplot(1, 2, 2)

plt.imshow(
    reconstruction_from_indices[
        0
    ].cpu().squeeze(),
    cmap="gray"
)

plt.title(
    "Decoded from Codebook Indices"
)

plt.axis("off")


plt.show()

In [ ]:
# ============================================================
# CELL 30: ENCODER OUTPUT VS QUANTIZED OUTPUT
# ============================================================

model.eval()

images, _ = next(iter(test_loader))

images = images[:1].to(device)


with torch.no_grad():

    (
        reconstruction,
        vq_loss,
        encoding_indices,
        codebook_loss,
        commitment_loss,
        z_e,
        z_q
    ) = model(images)


# Select one latent channel for visualization.

channel = 0


encoder_feature = (
    z_e[
        0,
        channel
    ]
    .cpu()
)


quantized_feature = (
    z_q[
        0,
        channel
    ]
    .cpu()
)


plt.figure(figsize=(5, 5))

plt.imshow(
    encoder_feature
)

plt.colorbar()

plt.title(
    "Continuous Encoder Feature"
)

plt.show()


plt.figure(figsize=(5, 5))

plt.imshow(
    quantized_feature
)

plt.colorbar()

plt.title(
    "Quantized Feature"
)

plt.show()

In [ ]:
# ============================================================
# CELL 31: CHECK THAT GRADIENTS FLOW
# ============================================================

model.train()

images, _ = next(iter(train_loader))

images = images.to(device)


optimizer.zero_grad()


(
    reconstruction,
    vq_loss,
    encoding_indices,
    codebook_loss,
    commitment_loss,
    z_e,
    z_q
) = model(images)


loss, reconstruction_loss = vqvae_loss(
    reconstruction,
    images,
    vq_loss
)


loss.backward()


# Inspect one encoder gradient
encoder_grad = (
    model.encoder.encoder[0]
    .weight
    .grad
)


# Inspect codebook gradient
codebook_grad = (
    model.quantizer.embedding
    .weight
    .grad
)


print(
    "Encoder gradient magnitude:",
    encoder_grad.abs().mean().item()
)

print(
    "Codebook gradient magnitude:",
    codebook_grad.abs().mean().item()
)

In [ ]:
# ============================================================
# CELL 32: MANUAL CHECK OF ONE LATENT VECTOR
# ============================================================

model.eval()

images, _ = next(iter(test_loader))

image = images[0:1].to(device)


with torch.no_grad():

    z_e = model.encoder(
        image
    )


# Choose one spatial location:
#
# row = 3
# col = 4

row = 3
col = 4


# Encoder vector at that position.
latent_vector = z_e[
    0,
    :,
    row,
    col
]


# Codebook:
#
# [K,D]

codebook = (
    model.quantizer.embedding.weight
)


# Compute Euclidean squared distance
# between this latent vector and every code.

distances = torch.sum(

    (
        codebook
        -
        latent_vector.unsqueeze(0)
    ) ** 2,

    dim=1
)


nearest_index = torch.argmin(
    distances
)


print(
    "Nearest codebook index:",
    nearest_index.item()
)

print(
    "Minimum squared distance:",
    distances[
        nearest_index
    ].item()
)

In [ ]:
# ============================================================
# CELL 33: WITH VS WITHOUT QUANTIZATION
# ============================================================

model.eval()

images, _ = next(iter(test_loader))

images = images[:10].to(device)


with torch.no_grad():

    # Encoder output
    z_e = model.encoder(
        images
    )


    # Decode WITHOUT quantization
    reconstruction_continuous = (
        model.decoder(
            z_e
        )
    )


    # Quantize
    (
        z_q,
        vq_loss,
        encoding_indices,
        codebook_loss,
        commitment_loss
    ) = model.quantizer(
        z_e
    )


    # Decode quantized representation
    reconstruction_quantized = (
        model.decoder(
            z_q
        )
    )

In [ ]:
# ============================================================
# CELL 34: DISPLAY QUANTIZATION EFFECT
# ============================================================

images_cpu = images.cpu()

continuous_cpu = (
    reconstruction_continuous.cpu()
)

quantized_cpu = (
    reconstruction_quantized.cpu()
)


plt.figure(figsize=(15, 6))


for i in range(10):

    # Original
    plt.subplot(
        3,
        10,
        i + 1
    )

    plt.imshow(
        images_cpu[i].squeeze(),
        cmap="gray"
    )

    plt.axis("off")


    # Continuous
    plt.subplot(
        3,
        10,
        i + 11
    )

    plt.imshow(
        continuous_cpu[i].squeeze(),
        cmap="gray"
    )

    plt.axis("off")


    # Quantized
    plt.subplot(
        3,
        10,
        i + 21
    )

    plt.imshow(
        quantized_cpu[i].squeeze(),
        cmap="gray"
    )

    plt.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 35: RANDOM CODEBOOK INDICES
# ============================================================

NUM_SAMPLES = 20


# Generate random integer code indices.
#
# Shape:
#
# [NUM_SAMPLES, 7, 7]

random_indices = torch.randint(

    low=0,

    high=NUM_EMBEDDINGS,

    size=(
        NUM_SAMPLES,
        7,
        7
    ),

    device=device
)


# Convert indices to embedding vectors.

random_quantized = (
    model.quantizer.embedding(
        random_indices
    )
)


# Shape:
#
# [B, 7, 7, D]
#
# Convert to:
#
# [B, D, 7, 7]

random_quantized = (
    random_quantized.permute(
        0,
        3,
        1,
        2
    )
)


with torch.no_grad():

    random_images = model.decoder(
        random_quantized
    )


random_images = random_images.cpu()


plt.figure(figsize=(10, 8))

for i in range(NUM_SAMPLES):

    plt.subplot(
        4,
        5,
        i + 1
    )

    plt.imshow(
        random_images[i].squeeze(),
        cmap="gray"
    )

    plt.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# CELL 36: IMPORT GMM AND PREPROCESSING TOOLS
# ============================================================

import numpy as np

from sklearn.mixture import GaussianMixture

from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt

import torch

In [ ]:
# ============================================================
# CELL 37: EXTRACT QUANTIZED LATENT VECTORS z_q
# ============================================================

# Put the VQ-VAE in evaluation mode.
model.eval()


# We will store one flattened latent vector for every image.
all_latents = []

# We also store labels.
#
# Labels are NOT required for the ordinary GMM,
# but they will be useful later for visualization
# and class-conditional experiments.
all_labels = []


with torch.no_grad():

    for images, labels in train_loader:

        images = images.to(device)

        # ----------------------------------------------------
        # FORWARD THROUGH VQ-VAE
        # ----------------------------------------------------

        (
            reconstruction,
            vq_loss,
            encoding_indices,
            codebook_loss,
            commitment_loss,
            z_e,
            z_q
        ) = model(images)


        # ----------------------------------------------------
        # z_q shape:
        #
        # [B, 64, 7, 7]
        # ----------------------------------------------------

        batch_size = z_q.size(0)


        # ----------------------------------------------------
        # FLATTEN EACH IMAGE'S LATENT MAP
        #
        # [B, 64, 7, 7]
        #
        # ->
        #
        # [B, 3136]
        # ----------------------------------------------------

        flattened_z_q = z_q.view(
            batch_size,
            -1
        )


        # Move from GPU to CPU.
        all_latents.append(
            flattened_z_q.cpu()
        )

        all_labels.append(
            labels.cpu()
        )


# ------------------------------------------------------------
# Concatenate all mini-batches
# ------------------------------------------------------------

all_latents = torch.cat(
    all_latents,
    dim=0
)

all_labels = torch.cat(
    all_labels,
    dim=0
)


print(
    "Latent dataset shape:",
    all_latents.shape
)

print(
    "Label shape:",
    all_labels.shape
)

In [ ]:
# ============================================================
# CELL 38: CONVERT LATENTS TO NUMPY
# ============================================================

latent_numpy = all_latents.numpy()

labels_numpy = all_labels.numpy()


print(
    "NumPy latent shape:",
    latent_numpy.shape
)

print(
    "Data type:",
    latent_numpy.dtype
)

In [ ]:
# ============================================================
# CELL 39: STANDARDIZE THE LATENT REPRESENTATIONS
# ============================================================

# StandardScaler computes, for every latent dimension:
#
#     mean_j
#     std_j
#
# and transforms:
#
#     z'_j = (z_j - mean_j) / std_j

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()


latent_scaled = scaler.fit_transform(
    latent_numpy
)


print(
    "Scaled latent shape:",
    latent_scaled.shape
)


# Check approximately:
#
# mean ~ 0
# std  ~ 1

print(
    "Overall approximate mean:",
    latent_scaled.mean()
)

print(
    "Overall approximate std:",
    latent_scaled.std()
)

In [ ]:
# ============================================================
# CELL 40: FIT THE GAUSSIAN MIXTURE MODEL
# ============================================================

# Number of Gaussian components.
#
# This is independent of the number of VQ codebook vectors.
#
# NUM_EMBEDDINGS = 128 refers to the VQ codebook.
#
# N_GMM_COMPONENTS = 20 refers to the number of Gaussian
# components used to approximate the DISTRIBUTION of complete
# latent maps.

N_GMM_COMPONENTS = 20


gmm = GaussianMixture(

    n_components=N_GMM_COMPONENTS,

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # Latent dimensionality is 3136.
    #
    # Full covariance would be extremely expensive.
    #
    # Use diagonal covariance:
    #
    # Sigma_k =
    #
    # diag(
    #   sigma_k1^2,
    #   sigma_k2^2,
    #   ...
    # )
    # --------------------------------------------------------

    covariance_type="diag",

    # Initialization method
    init_params="kmeans",

    # EM maximum iterations
    max_iter=200,

    # Stop when log-likelihood improvement becomes small
    tol=1e-3,

    # Small regularization prevents variances
    # from becoming numerically zero.
    reg_covar=1e-6,

    random_state=42,

    verbose=1
)


# ------------------------------------------------------------
# FIT USING EM
# ------------------------------------------------------------
#
# This learns:
#
#     mixing coefficients pi_k
#
#     means mu_k
#
#     variances Sigma_k
#
# by maximizing the likelihood of latent representations.
# ------------------------------------------------------------

gmm.fit(
    latent_scaled
)

In [ ]:
# ============================================================
# CELL 41: INSPECT GMM PARAMETERS
# ============================================================

print(
    "Converged:",
    gmm.converged_
)

print(
    "Number of EM iterations:",
    gmm.n_iter_
)

print(
    "GMM weights shape:",
    gmm.weights_.shape
)

print(
    "GMM means shape:",
    gmm.means_.shape
)

print(
    "GMM covariance shape:",
    gmm.covariances_.shape
)

In [ ]:
# ============================================================
# CELL 42: PRINT GMM MIXING COEFFICIENTS
# ============================================================

for k, weight in enumerate(
    gmm.weights_
):

    print(
        f"Component {k:02d}: "
        f"pi = {weight:.4f}"
    )

In [ ]:
# ============================================================
# CELL 43: PLOT GMM COMPONENT PROBABILITIES
# ============================================================

plt.figure(figsize=(10, 5))

plt.bar(
    np.arange(N_GMM_COMPONENTS),
    gmm.weights_
)

plt.xlabel(
    "GMM Component"
)

plt.ylabel(
    "Mixing Probability"
)

plt.title(
    "Learned GMM Mixing Coefficients"
)

plt.show()

In [ ]:
# ============================================================
# CELL 44: GMM COMPONENT ASSIGNMENTS
# ============================================================

gmm_assignments = gmm.predict(
    latent_scaled
)


print(
    "Assignment shape:",
    gmm_assignments.shape
)


print(
    "First 20 assignments:"
)

print(
    gmm_assignments[:20]
)

In [ ]:
# ============================================================
# CELL 45: DIGIT DISTRIBUTION WITHIN EACH GMM COMPONENT
# ============================================================

for component in range(
    N_GMM_COMPONENTS
):

    # Select training examples assigned to this component.
    mask = (
        gmm_assignments
        ==
        component
    )


    component_labels = (
        labels_numpy[mask]
    )


    print(
        f"\nGMM Component {component}"
    )

    print(
        "Number of samples:",
        len(component_labels)
    )


    if len(component_labels) > 0:

        counts = np.bincount(
            component_labels,
            minlength=10
        )

        proportions = (
            counts
            /
            counts.sum()
        )


        for digit in range(10):

            print(
                f"Digit {digit}: "
                f"{proportions[digit]:.3f}"
            )

In [ ]:
# ============================================================
# CELL 46: SAMPLE NEW LATENT VECTORS FROM THE GMM
# ============================================================

NUM_NEW_SAMPLES = 25


# gmm.sample() returns:
#
# sampled_latents
#     Shape: [NUM_NEW_SAMPLES, 3136]
#
# sampled_components
#     Which Gaussian component generated each sample.

sampled_latents_scaled, sampled_components = (
    gmm.sample(
        NUM_NEW_SAMPLES
    )
)


print(
    "Sampled latent shape:",
    sampled_latents_scaled.shape
)

print(
    "GMM components used:"
)

print(
    sampled_components
)

In [ ]:
# ============================================================
# CELL 47: INVERSE STANDARDIZATION
# ============================================================

sampled_latents = scaler.inverse_transform(
    sampled_latents_scaled
)


print(
    "Recovered latent shape:",
    sampled_latents.shape
)

In [ ]:
# ============================================================
# CELL 48: NUMPY -> PYTORCH
# ============================================================

sampled_latents_tensor = torch.tensor(

    sampled_latents,

    dtype=torch.float32,

    device=device
)


print(
    sampled_latents_tensor.shape
)

In [ ]:
# ============================================================
# CELL 49: RESTORE SPATIAL LATENT SHAPE
# ============================================================

sampled_latent_maps = (
    sampled_latents_tensor.view(
        NUM_NEW_SAMPLES,
        EMBEDDING_DIM,
        7,
        7
    )
)


print(
    "Latent map shape:",
    sampled_latent_maps.shape
)

In [ ]:
# ============================================================
# CELL 50: DIRECT GMM LATENT DECODING
# ============================================================

model.eval()


with torch.no_grad():

    generated_direct = model.decoder(
        sampled_latent_maps
    )


generated_direct = (
    generated_direct.cpu()
)


print(
    "Generated image shape:",
    generated_direct.shape
)

In [ ]:
# ============================================================
# CELL 51: DISPLAY DIRECT GMM GENERATIONS
# ============================================================

plt.figure(figsize=(8, 8))


for i in range(
    NUM_NEW_SAMPLES
):

    plt.subplot(
        5,
        5,
        i + 1
    )

    plt.imshow(
        generated_direct[
            i
        ].squeeze(),
        cmap="gray"
    )

    plt.title(
        f"GMM {sampled_components[i]}"
    )

    plt.axis("off")


plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# CELL 52: PROJECT GMM LATENTS TO NEAREST VQ CODEBOOK VECTORS
# ============================================================

def quantize_external_latents(
    latent_maps,
    model
):

    """
    latent_maps shape:

        [B, D, H, W]

    This function takes arbitrary continuous latent maps,
    such as GMM samples, and replaces every spatial vector
    with its nearest VQ-VAE codebook embedding.

    Returns:

        quantized_maps
            [B, D, H, W]

        indices
            [B, H, W]
    """


    B, D, H, W = latent_maps.shape


    # --------------------------------------------------------
    # Rearrange:
    #
    # [B,D,H,W]
    #
    # ->
    #
    # [B,H,W,D]
    # --------------------------------------------------------

    latents = latent_maps.permute(
        0,
        2,
        3,
        1
    ).contiguous()


    # --------------------------------------------------------
    # Flatten spatial positions:
    #
    # [B,H,W,D]
    #
    # ->
    #
    # [B*H*W,D]
    # --------------------------------------------------------

    flat_latents = latents.view(
        -1,
        D
    )


    # --------------------------------------------------------
    # VQ codebook:
    #
    # [NUM_EMBEDDINGS,D]
    # --------------------------------------------------------

    codebook = (
        model.quantizer
        .embedding
        .weight
    )


    # --------------------------------------------------------
    # Compute:
    #
    # ||z-e||^2
    #
    # =
    #
    # ||z||^2
    # +
    # ||e||^2
    # -
    # 2 z^T e
    # --------------------------------------------------------

    distances = (

        torch.sum(
            flat_latents ** 2,
            dim=1,
            keepdim=True
        )

        +

        torch.sum(
            codebook ** 2,
            dim=1
        )

        -

        2
        *
        torch.matmul(
            flat_latents,
            codebook.t()
        )
    )


    # --------------------------------------------------------
    # Find nearest codebook index.
    # --------------------------------------------------------

    nearest_indices = torch.argmin(
        distances,
        dim=1
    )


    # --------------------------------------------------------
    # Retrieve corresponding code vectors.
    # --------------------------------------------------------

    quantized = model.quantizer.embedding(
        nearest_indices
    )


    # --------------------------------------------------------
    # Restore:
    #
    # [B,H,W,D]
    # --------------------------------------------------------

    quantized = quantized.view(
        B,
        H,
        W,
        D
    )


    # --------------------------------------------------------
    # Convert:
    #
    # [B,H,W,D]
    #
    # ->
    #
    # [B,D,H,W]
    # --------------------------------------------------------

    quantized = quantized.permute(
        0,
        3,
        1,
        2
    ).contiguous()


    # Also return the code-index map.
    nearest_indices = nearest_indices.view(
        B,
        H,
        W
    )


    return (
        quantized,
        nearest_indices
    )

In [ ]:
# ============================================================
# CELL 53: GMM LATENT -> NEAREST CODEBOOK
# ============================================================

with torch.no_grad():

    (
        sampled_quantized_maps,
        sampled_code_indices
    ) = quantize_external_latents(

        sampled_latent_maps,

        model
    )


print(
    "Quantized latent shape:",
    sampled_quantized_maps.shape
)

print(
    "Code index shape:",
    sampled_code_indices.shape
)


In [ ]:
# ============================================================
# CELL 54: DECODE GMM + VQ SAMPLES
# ============================================================

model.eval()


with torch.no_grad():

    generated_quantized = model.decoder(
        sampled_quantized_maps
    )


generated_quantized = (
    generated_quantized.cpu()
)

In [ ]:
# ============================================================
# CELL 55: DISPLAY GMM + VQ GENERATED IMAGES
# ============================================================

plt.figure(figsize=(8, 8))


for i in range(
    NUM_NEW_SAMPLES
):

    plt.subplot(
        5,
        5,
        i + 1
    )

    plt.imshow(
        generated_quantized[
            i
        ].squeeze(),
        cmap="gray"
    )

    plt.title(
        f"GMM {sampled_components[i]}"
    )

    plt.axis("off")


plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# CELL 56: DIRECT GMM VS GMM + VQ
# ============================================================

plt.figure(
    figsize=(15, 6)
)


for i in range(10):

    # --------------------------------------------------------
    # ROW 1:
    #
    # Directly decoded GMM sample
    # --------------------------------------------------------

    plt.subplot(
        2,
        10,
        i + 1
    )

    plt.imshow(
        generated_direct[
            i
        ].squeeze(),
        cmap="gray"
    )

    plt.axis("off")

    if i == 0:

        plt.ylabel(
            "GMM Direct"
        )


    # --------------------------------------------------------
    # ROW 2:
    #
    # GMM sample projected to codebook
    # --------------------------------------------------------

    plt.subplot(
        2,
        10,
        i + 11
    )

    plt.imshow(
        generated_quantized[
            i
        ].squeeze(),
        cmap="gray"
    )

    plt.axis("off")

    if i == 0:

        plt.ylabel(
            "GMM + VQ"
        )


plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# CELL 57: SHOW ONE GENERATED DISCRETE CODE MAP
# ============================================================

sample_number = 0


code_map = (
    sampled_code_indices[
        sample_number
    ]
    .cpu()
    .numpy()
)


print(
    "Code indices:"
)

print(
    code_map
)


plt.figure(
    figsize=(5, 5)
)

plt.imshow(
    code_map,
    interpolation="nearest"
)

plt.colorbar(
    label="Codebook index"
)

plt.title(
    "Discrete Latent Map Generated from GMM"
)

plt.xlabel(
    "Latent width"
)

plt.ylabel(
    "Latent height"
)

plt.show()

In [ ]:
# ============================================================
# CELL 58: SAMPLE FROM ONE PARTICULAR GMM COMPONENT
# ============================================================

COMPONENT_ID = 0

NUM_COMPONENT_SAMPLES = 20


# ------------------------------------------------------------
# Mean of selected Gaussian
#
# Shape:
#
# [3136]
# ------------------------------------------------------------

component_mean = gmm.means_[
    COMPONENT_ID
]


# ------------------------------------------------------------
# Because covariance_type="diag":
#
# covariances_[k]
#
# contains VARIANCES, not standard deviations.
# ------------------------------------------------------------

component_variance = gmm.covariances_[
    COMPONENT_ID
]


component_std = np.sqrt(
    component_variance
)


# ------------------------------------------------------------
# epsilon ~ N(0,I)
# ------------------------------------------------------------

epsilon = np.random.randn(
    NUM_COMPONENT_SAMPLES,
    latent_scaled.shape[1]
)


# ------------------------------------------------------------
# z = mu + sigma * epsilon
#
# These samples are still in SCALED coordinates.
# ------------------------------------------------------------

component_samples_scaled = (

    component_mean[None, :]

    +

    component_std[None, :]

    *
    epsilon
)

In [ ]:
# ============================================================
# CELL 59: CONVERT COMPONENT SAMPLES TO VQ LATENT MAPS
# ============================================================

component_samples = scaler.inverse_transform(
    component_samples_scaled
)


component_tensor = torch.tensor(

    component_samples,

    dtype=torch.float32,

    device=device
)


component_maps = component_tensor.view(

    NUM_COMPONENT_SAMPLES,

    EMBEDDING_DIM,

    7,

    7
)

In [ ]:
# ============================================================
# CELL 60: GENERATE IMAGES FROM ONE GMM COMPONENT
# ============================================================

with torch.no_grad():

    component_quantized, component_indices = (
        quantize_external_latents(
            component_maps,
            model
        )
    )


    component_images = model.decoder(
        component_quantized
    )


component_images = component_images.cpu()

In [ ]:
# ============================================================
# CELL 61: DISPLAY ONE GMM COMPONENT
# ============================================================

plt.figure(figsize=(10, 8))


for i in range(
    NUM_COMPONENT_SAMPLES
):

    plt.subplot(
        4,
        5,
        i + 1
    )

    plt.imshow(
        component_images[
            i
        ].squeeze(),
        cmap="gray"
    )

    plt.axis("off")


plt.suptitle(
    f"Samples from GMM Component {COMPONENT_ID}"
)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# CELL 62: SELECT NUMBER OF GMM COMPONENTS USING BIC
# ============================================================

# Use a subset to keep this experiment manageable.
SUBSET_SIZE = 10000


rng = np.random.default_rng(
    42
)


subset_indices = rng.choice(

    latent_scaled.shape[0],

    size=SUBSET_SIZE,

    replace=False
)


latent_subset = latent_scaled[
    subset_indices
]


component_choices = [
    5,
    10,
    15,
    20,
    30
]


bic_scores = []


for n_components in component_choices:

    print(
        "\nFitting:",
        n_components,
        "components"
    )


    temp_gmm = GaussianMixture(

        n_components=n_components,

        covariance_type="diag",

        max_iter=100,

        random_state=42,

        reg_covar=1e-6
    )


    temp_gmm.fit(
        latent_subset
    )


    bic_value = temp_gmm.bic(
        latent_subset
    )


    bic_scores.append(
        bic_value
    )


    print(
        "BIC:",
        bic_value
    )

In [ ]:
# ============================================================
# CELL 63: PLOT BIC
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    component_choices,
    bic_scores,
    marker="o"
)

plt.xlabel(
    "Number of GMM Components"
)

plt.ylabel(
    "BIC"
)

plt.title(
    "Selecting Number of GMM Components"
)

plt.show()

In [ ]:
# ============================================================
# CELL 64: COMPLETE GMM -> VQ -> DECODER GENERATION FUNCTION
# ============================================================

def generate_from_gmm(
    gmm,
    scaler,
    model,
    num_samples,
    embedding_dim,
    device,
    requantize=True
):

    """
    Generate new samples from the fitted GMM.

    Pipeline:

        GMM
          ->
        latent vector
          ->
        inverse standardization
          ->
        reshape to spatial VQ latent map
          ->
        optionally project onto nearest VQ codes
          ->
        decoder
          ->
        generated image


    Parameters
    ----------
    gmm:
        Fitted sklearn GaussianMixture.

    scaler:
        Fitted StandardScaler used before GMM fitting.

    model:
        Trained VQ-VAE.

    num_samples:
        Number of images to generate.

    embedding_dim:
        VQ embedding dimension.

    requantize:
        If True:
            GMM -> nearest VQ codes -> decoder.

        If False:
            GMM -> decoder directly.
    """


    # --------------------------------------------------------
    # STEP 1:
    # Sample standardized latent vectors from the GMM.
    # --------------------------------------------------------

    sampled_scaled, components = gmm.sample(
        num_samples
    )


    # --------------------------------------------------------
    # STEP 2:
    # Undo standardization.
    # --------------------------------------------------------

    sampled = scaler.inverse_transform(
        sampled_scaled
    )


    # --------------------------------------------------------
    # STEP 3:
    # NumPy -> PyTorch
    # --------------------------------------------------------

    sampled_tensor = torch.tensor(

        sampled,

        dtype=torch.float32,

        device=device
    )


    # --------------------------------------------------------
    # STEP 4:
    # Flattened latent -> spatial latent map
    # --------------------------------------------------------

    latent_maps = sampled_tensor.view(

        num_samples,

        embedding_dim,

        7,

        7
    )


    # --------------------------------------------------------
    # STEP 5:
    # Optional VQ projection.
    # --------------------------------------------------------

    if requantize:

        latent_maps, indices = (
            quantize_external_latents(
                latent_maps,
                model
            )
        )

    else:

        indices = None


    # --------------------------------------------------------
    # STEP 6:
    # Decode.
    # --------------------------------------------------------

    model.eval()

    with torch.no_grad():

        images = model.decoder(
            latent_maps
        )


    return (
        images.cpu(),
        components,
        indices
    )

In [ ]:
# ============================================================
# CELL 65: GENERATE NEW SAMPLES
# ============================================================

new_images, components, new_indices = generate_from_gmm(

    gmm=gmm,

    scaler=scaler,

    model=model,

    num_samples=25,

    embedding_dim=EMBEDDING_DIM,

    device=device,

    requantize=True
)

In [ ]:
# ============================================================
# CELL 66: DISPLAY FINAL GMM + VQ-VAE GENERATIONS
# ============================================================

plt.figure(figsize=(8, 8))


for i in range(25):

    plt.subplot(
        5,
        5,
        i + 1
    )

    plt.imshow(
        new_images[i].squeeze(),
        cmap="gray"
    )

    plt.title(
        f"C{components[i]}"
    )

    plt.axis("off")


plt.tight_layout()

plt.show()